In [0]:
spark.table("workspace.logistics_project.maintenance_records").printSchema()

root
 |-- maintenance_id: string (nullable = true)
 |-- truck_id: string (nullable = true)
 |-- maintenance_date: date (nullable = true)
 |-- maintenance_type: string (nullable = true)
 |-- odometer_reading: long (nullable = true)
 |-- labor_hours: double (nullable = true)
 |-- labor_cost: double (nullable = true)
 |-- parts_cost: double (nullable = true)
 |-- total_cost: double (nullable = true)
 |-- facility_location: string (nullable = true)
 |-- downtime_hours: double (nullable = true)
 |-- service_description: string (nullable = true)



In [0]:
#Read Tables
import pyspark.sql.functions as F

maintenance = spark.table("workspace.logistics_project.maintenance_records")
trucks = spark.table("workspace.logistics_project.trucks")
truck_utilization = spark.table("workspace.logistics_project.truck_utilization_metrics")

In [0]:
#Join Tables
maintenance_analysis = (
    maintenance
    .join(trucks, "truck_id", "inner")
    .join(
        truck_utilization.select(
            "truck_id",
            "utilization_rate",
            "total_miles"
        ),
        "truck_id",
        "left"
    )
)

In [0]:
#Calculate Maintenance KPIs
maintenance_metrics = (
    maintenance_analysis
    .groupBy(
        "truck_id",
        "unit_number",
        "make",
        "model_year",
        "fuel_type"
    )
    .agg(
        F.count("maintenance_id").alias("maintenance_events"),
        F.round(F.sum("labor_cost"),2).alias("total_labor_cost"),
        F.round(F.sum("parts_cost"),2).alias("total_parts_cost"),
        F.round(F.sum("total_cost"),2).alias("total_maintenance_cost"),
        F.round(F.sum("downtime_hours"),2).alias("total_downtime_hours"),
        F.round(F.avg("utilization_rate"),2).alias("avg_utilization_rate"),
        F.max("total_miles").alias("total_miles")
    )
)

In [0]:
#Cost Per Event
maintenance_metrics = maintenance_metrics.withColumn(
    "cost_per_event",
    F.round(
        F.col("total_maintenance_cost") /
        F.col("maintenance_events"),
        2
    )
)

In [0]:
#Maintenance Cost Per Mile
maintenance_metrics = maintenance_metrics.withColumn(
    "maintenance_cost_per_mile",
    F.round(
        F.col("total_maintenance_cost") /
        F.col("total_miles"),
        4
    )
)

In [0]:
#View Results
display(
    maintenance_metrics.orderBy(
        F.col("total_maintenance_cost").desc()
    )
)

truck_id,unit_number,make,model_year,fuel_type,maintenance_events,total_labor_cost,total_parts_cost,total_maintenance_cost,total_downtime_hours,avg_utilization_rate,total_miles,cost_per_event,maintenance_cost_per_mile
TRK00073,2680,Mack,2016,Diesel,1404,580087.08,2176576.56,2756663.64,35866.8,0.85,53715,1963.44,51.3202
TRK00099,1461,Freightliner,2015,Diesel,1224,547705.8,2088637.92,2636343.72,26773.2,0.82,51767,2153.88,50.9271
TRK00014,9624,International,2015,Diesel,1224,664381.8,1920483.72,2584865.52,33854.4,0.81,47022,2111.82,54.9714
TRK00026,4145,International,2021,Diesel,1152,516890.16,2028615.12,2545505.28,28364.4,0.82,57767,2209.64,44.065
TRK00044,8170,Volvo,2015,Diesel,1368,648118.08,1865590.2,2513708.28,38894.4,0.89,54829,1837.51,45.8463
TRK00038,3147,International,2015,Diesel,1116,457901.28,2025707.76,2483609.04,28670.4,0.83,55718,2225.46,44.5746
TRK00040,3527,International,2015,Diesel,936,472401.36,1959677.28,2432078.64,23943.6,0.78,61165,2598.37,39.7626
TRK00033,9317,Mack,2015,Diesel,1152,480331.8,1950187.68,2430519.48,32695.2,0.83,55757,2109.83,43.5913
TRK00052,4833,Freightliner,2015,Diesel,1116,401868.36,1893391.2,2295259.56,26542.8,0.81,50654,2056.68,45.3125
TRK00071,8873,Mack,2017,Diesel,1116,451846.8,1769229.0,2221075.8,27414.0,0.83,51935,1990.21,42.7665


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
#Create Gold Table
spark.sql("""
SHOW TABLES IN workspace.logistics_gold
""").show(truncate=False)

+--------------+--------------------+-----------+
|database      |tableName           |isTemporary|
+--------------+--------------------+-----------+
|logistics_gold|driver_performance  |false      |
|logistics_gold|fleet_utilization   |false      |
|logistics_gold|maintenance_analysis|false      |
|logistics_gold|route_profitability |false      |
+--------------+--------------------+-----------+



In [0]:
maintenance_metrics.write \
.format("delta") \
.saveAsTable(
    "workspace.logistics_gold.maintenance_analysis"
)

In [0]:
#MERGE
from delta.tables import DeltaTable

gold_table = DeltaTable.forName(
    spark,
    "workspace.logistics_gold.maintenance_analysis"
)

gold_table.alias("target").merge(
    maintenance_metrics.alias("source"),
    "target.truck_id = source.truck_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
#Verify
display(
    spark.table(
        "workspace.logistics_gold.maintenance_analysis"
    )
)

truck_id,unit_number,make,model_year,fuel_type,maintenance_events,total_labor_cost,total_parts_cost,total_maintenance_cost,total_downtime_hours,avg_utilization_rate,total_miles,cost_per_event,maintenance_cost_per_mile
TRK00070,1854,Kenworth,2015,Diesel,540,294943.32,1006180.2,1301123.52,11077.2,0.86,60809,2409.49,21.3969
TRK00065,7859,Freightliner,2015,Diesel,936,429841.44,1640756.16,2070597.6,21232.8,0.84,58599,2212.18,35.335
TRK00006,6082,Kenworth,2017,Diesel,936,394115.04,1433994.84,1828109.88,21157.2,0.8,50000,1953.11,36.5622
TRK00025,8967,Freightliner,2018,Diesel,828,375222.96,1211733.36,1586956.32,20523.6,0.88,56399,1916.61,28.138
TRK00013,3093,Volvo,2018,Diesel,1224,520238.16,1566003.6,2086241.76,31266.0,0.81,51938,1704.45,40.1679
TRK00111,1086,Kenworth,2015,Diesel,1008,450824.04,1584352.8,2035176.84,22150.8,0.85,57710,2019.02,35.2656
TRK00014,9624,International,2015,Diesel,1224,664381.8,1920483.72,2584865.52,33854.4,0.81,47022,2111.82,54.9714
TRK00083,8167,Kenworth,2015,Diesel,360,162170.64,266095.44,428266.08,8924.4,0.82,48467,1189.63,8.8362
TRK00003,2335,Peterbilt,2018,Diesel,41,18587.94,71573.48,90161.42,1133.1,null,null,2199.06,null
TRK00044,8170,Volvo,2015,Diesel,1368,648118.08,1865590.2,2513708.28,38894.4,0.89,54829,1837.51,45.8463
